<!-- nav -->
[← TypeScript tutorial](README.md) · [Home](../../README.md) · [2 · Finding what matches →](02_Finding_What_Matches.ipynb)

# 1 · Constraints your data must keep

**The problem.** An address book has constraints: a contact is an adult, every phone has a number. Today they live in a
validation function in the TypeScript service, again in the Python import job, and again in a sentence in the interface
document. The three drift apart. Someone tightens the age constraint in one place, and the others go on accepting what
it now refuses.

mbse-patterns writes such a constraint once, as **data**: a *predicate*, kept beside the schemas, stored and sent like
any other data, and applied the same way by every program and language that loads it. This case study writes two
constraints, checks data against them, and saves them.

The address book is mbse-schemas' (its tutorial, case study 2): contacts listed in a directory, each with phones. Its
schemas are in `toolkit.ts`, beside this notebook.

In [1]:
import { Expressions as E } from "@mbse/expressions";
import { Constraints, Predicates, Validators } from "@mbse/patterns";
import { JSON as SchemaJSON } from "@mbse/schemas/Framework";
import { Contact, Directory, Phone, book, listed } from "./toolkit.ts";

const store = book();
const ann = store.Contact().name("Ann").age(34n)
  .phones((e: any) => e.label("home").phone((p: any) => p.number("555-0100"))).create();
const bob = store.Contact().name("Bob").phones((e: any) => e.label("work").phone((p: any) => p)).create(); // no age; a phone, no number
const kid = store.Contact().name("Kid").age(9n).create();
listed(store, ann, bob, kid);

## Step 1: a predicate

A predicate has a name, **symbols**, each bound to an object of a schema, and a **constraint** about them. "A contact is
an adult" has one symbol, `the`, a `Contact`, and the constraint `the.age >= 18`. It's built fluently, as a schema is.
The constraint is written with mbse-expressions' writers: `the.age` reads a property, and `.ge(18n)` compares it
(integers are `bigint`s):

In [2]:
const the = E.variable("the");
const IsAnAdult = new Predicates.OfPredicate.Builder()
  .name("IsAnAdult")
  .description("18 or older")
  .symbols({ the: Contact })
  .requires(the.age.ge(18n))
  .create();
console.log(IsAnAdult.name, [...IsAnAdult.symbols.keys()], IsAnAdult.requires.name);

IsAnAdult [ "the" ] ge


The constraint is an expression of mbse-expressions' Basic dialect, `ge(get(the, 'age'), 18)`. Nothing here is
TypeScript any more, which is what lets Python apply the same constraint.

## Step 2: checking data

`Validators.Validate(store, predicates)` checks the predicates on the objects you give it. A predicate applies to a
**match**, a binding of each symbol to an object of its schema; with one symbol, each contact is a match.

In [3]:
const validate = Validators.Validate(store, [IsAnAdult]);
console.log(validate(Contact, ann), validate(Contact, kid), validate(Contact, bob));

[] [ "the=Contact#0: 'IsAnAdult' does not hold" ] [ "the=Contact#0: 'IsAnAdult' is unknown" ]


Ann holds; the kid doesn't. Bob's answer is neither: his age isn't on file, so whether he is 18 or older is
**unknown**, and the validator says so rather than guessing. A plain function would have compared `undefined` and
failed him. You decide what unknown means for each use:

In [4]:
console.log(Validators.Validate(store, [IsAnAdult], { unknown: "ignore" })(Contact, bob),
  Validators.Validate(store, [IsAnAdult], { unknown: "violation" })(Contact, bob));

[] [ "the=Contact#0: 'IsAnAdult' does not hold" ]


`Reachable` checks every match among what a root reaches, here the whole directory, and labels each problem by the
object it's about, as mbse-schemas' validator does:

In [5]:
console.log(validate.Reachable(Directory, store.singleton("book.Directory")));

[
  "the=Contact#2: 'IsAnAdult' is unknown",
  "the=Contact#3: 'IsAnAdult' does not hold"
]


## Step 3: a constraint about two objects

"Every phone a contact owns has a number" is about two objects at once: a contact `c` and a phone `p`. With several
symbols, the matches are every **combination** of their objects, so the constraint must say how they're related: that
`p` is one of `c`'s phones. `Predicates.Contains(c.phones, ...)` says it, calling its function with a variable for each
entry of `c`'s `phones`:

In [6]:
const [c, p] = [E.variable("c"), E.variable("p")];
const OwnedNumbered = new Predicates.OfPredicate.Builder()
  .name("OwnedNumbered")
  .symbols({ c: Contact, p: Phone })
  .requires(E.operation("implies", Predicates.Contains(c.phones, (e) => e.phone.eq(p)), p.has("number")))
  .create();
console.log(Validators.Validate(store, [IsAnAdult, OwnedNumbered]).Reachable(Directory, store.singleton("book.Directory")));

[
  "the=Contact#2: 'IsAnAdult' is unknown",
  "the=Contact#3: 'IsAnAdult' does not hold",
  "c=Contact#2, p=Phone#5: 'OwnedNumbered' does not hold"
]


Bob's phone has no number. Every other pair of a contact and a phone, Ann and Bob's phone say, satisfies `implies`
trivially, because the phone isn't hers.

## Step 4: constraints that can't be right are refused up front

A validator checks its predicates when it's made, so a constraint that could never work is reported once, not once per
match:

In [7]:
const Broken = new Predicates.OfPredicate.Builder().name("Broken").symbols({ c: Contact }).requires(c.age.ge(E.variable("n"))).create();
try {
  Validators.Validate(store, [Broken]);
} catch (error) {
  console.log(String(error));
}

ValueError: predicate 'Broken': requires: argument 1: variable 'n' is not bound


## Step 5: constraints are data

`Constraints.check` gathers predicates in a set. A set saves like any object graph: `JSON.ToJSON` writes it, with each
symbol's schema by name, so any program that has the schemas can read the constraints back:

In [8]:
const constraints = Constraints.check([IsAnAdult, OwnedNumbered]);
const text = SchemaJSON.ToJSON(Constraints.Builders).Reachable(Constraints.OfSet.Schema, constraints);
console.log(text.slice(0, 400), "...");

{"root": "s0", "objects": {"s0": {"kind": "set", "arguments": [{"argument": {"$ref": "s1", "$schema": "Patterns.Predicate"}, "index": 0}, {"argument": {"$ref": "s2", "$schema": "Patterns.Predicate"}, "index": 1}]}, "s1": {"kind": "predicate", "name": "IsAnAdult", "description": "18 or older", "symbols": [{"name": "the", "type": {"named": {"name": "Contact"}}}], "arguments": [{"argument": {"$ref":  ...


In [9]:
const copy = SchemaJSON.FromJSON(new Constraints.OfStore(store)).Reachable(Constraints.OfSet.Schema, text) as Constraints.OfSet;
console.log(copy.predicates.map((x: any) => x.name), copy.predicates[0].symbols.get("the") === Contact);

[ "IsAnAdult", "OwnedNumbered" ] true


Reading resolves `Contact` by name in the store you give it (`new Constraints.OfStore(store)`), so the copy's symbols
are your schemas, not lookalikes. Python reads the same text and gets the same constraints.

## Why it works this way

- **Constraints beside the schemas, not inside them.** Several sets of constraints can apply to one schema: what the
  editing form insists on isn't what the archive tolerates. A schema stays a description of data; a program chooses its
  constraints.
- **Unknown is not false.** Absent isn't `undefined` in mbse-schemas, and a constraint about an absent property has no
  answer. Treating it as false silently fails good records; treating it as true silently passes bad ones.
- **Matches, not loops.** A predicate with two symbols applies to every pair. You don't write the loop; you say how the
  objects are related, and the next case study shows that this is also what makes the constraint fast to query.

<!-- nav -->
[← TypeScript tutorial](README.md) · [Home](../../README.md) · [2 · Finding what matches →](02_Finding_What_Matches.ipynb)